# Modelo GCN

En este notebook entrenamos un modelo de deep learning para plantear algo que los otros modelos no capturan: que la velocidad de un segmento depende de sus segmentos vecinos en la red de calles, no solo de su pasado.

Usamos un modelo Graph Convolutional Network (GCN). Para cada hora, construimos un grafo con nuestros segmentos como nodos, y cada nodo lleva como característica su `speed_lag1`,`hora`, `clima`, etc. La diferencia con el modelo LightGBM es que las capas de GCN agregan información de los segmentos vecinos antes de predecir, por lo tanto la predicción de un segmento depende de lo que pasa en los segmentos conectados a él y no solo de su histórico.

El enfoque principal para esto (STGCN, DCRNN) combina convoluciones de grafo con módulos recurrentes de secuencia temporal, pero es bastante más complejo de implementar. Por lo tanto esta versión es más simple ,el componente temporal lo aporta el lag, el componente espacial la GCN.

El procedimiento es el siguiente:
1. Construir el grafo de adyacencia entre nuestros segmentos (quién es vecino de quién).
2. Preparar los datos en formato "snapshot por hora".
3. Definir y entrenar el modelo.
4. Evaluar y comparar contra LightGBM, con las mismas métricas.



## 0. Importación de paquetes

In [3]:
!pip install torch_geometric --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.2 MB/s eta 0:00:00


In [4]:
from google.colab import drive
drive.mount('/content/drive')

data_clean = "/content/drive/MyDrive/TFM/data_clean"

import pandas as pd
import numpy as np
from collections import Counter
import duckdb
import random
import time
import json
import torch

from torch_geometric.data import Data
from torch.utils.data import Dataset
from torch_geometric.loader import DataLoader

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Grafo de adyacencia entre segmentos

Consideramos dos segmentos vecinos si uno de sus extremos está muy cerca del extremo del otro.

Para ello comparamos la geometría de los segmentos. Para cada uno miramos las 4 combinaciones posibles entre sus dos extremos y los de todos los demás y si alguna distancia es menor que 60 metros (margen para el ruido GPS) los marcamos como vecinos.

In [5]:
# cargamos dim_segmentos
dim_segmentos = pd.read_parquet(f"{data_clean}/dim_segmentos.parquet")

In [6]:
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371000
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

umbral = 60

start_lat = dim_segmentos["start_latitude"].values
start_lon = dim_segmentos["start_longitude"].values
end_lat = dim_segmentos["end_latitude"].values
end_lon = dim_segmentos["end_longitude"].values
ids = dim_segmentos["segment_id"].values

adyacencia = set()
for i in range(len(dim_segmentos)):
    dist_end_to_start = haversine_np(end_lat[i], end_lon[i], start_lat, start_lon)
    dist_start_to_end = haversine_np(start_lat[i], start_lon[i], end_lat, end_lon)
    dist_start_to_start = haversine_np(start_lat[i], start_lon[i], start_lat, start_lon)
    dist_end_to_end = haversine_np(end_lat[i], end_lon[i], end_lat, end_lon)

    conectados = np.where(
        (dist_end_to_start < umbral) |
        (dist_start_to_end < umbral) |
        (dist_start_to_start < umbral) |
        (dist_end_to_end < umbral)
    )[0]

    for j in conectados:
        if ids[j] != ids[i]:
            adyacencia.add((ids[i], ids[j]))
            adyacencia.add((ids[j], ids[i]))

print(f"Conexiones encontradas: {len(adyacencia)}")

grados = Counter()
for a, b in adyacencia:
    grados[a] += 1

grados_valores = list(grados.values())
print(f"Segmentos sin ningún vecino: {len(dim_segmentos) - len(grados)}")
print(f"Media de vecinos: {sum(grados_valores)/len(grados_valores):.2f}")
print(f"Distribución: min={min(grados_valores)}, max={max(grados_valores)}")

Conexiones encontradas: 6864
Segmentos sin ningún vecino: 3
Media de vecinos: 6.58
Distribución: min=1, max=13


Conseguimos 6.864 conexiones, una media de 6.58 vecinos por segmento, y solo 3 segmentos aislados.

## 2. Preparar los snapshots por hora

A diferencia de LightGBM, donde cada fila es independiente, aquí cada snapshot necesita todos los nodos a la vez porque el modelo propaga información entre vecinos. Para cada hora, construimos una matriz con las características de todos los segmentos y el grafo de conexiones entre ellos.

### 2.1. Nodos y conexiones

Nos quedamos solo con los segmentos que tienen al menos un vecino (los 3 aislados no nos aportan nada) y les asignamos un índice numérico, ya que las GNN trabajan con índices de nodo y no con nuestros `segment_id` originales.

In [7]:
nodos_con_vecinos = sorted(grados.keys())
node_to_idx = {seg_id: i for i, seg_id in enumerate(nodos_con_vecinos)}
n_nodos = len(nodos_con_vecinos)

print(f"Número de nodos en la GCN: {n_nodos}")

Número de nodos en la GCN: 1043


PyTorch Geometric espera las conexiones como una matriz de 2 filas: la primera con los nodos "origen" y la segunda con los "destino" de cada conexión.

In [8]:
edge_index_list = [
    [node_to_idx[a], node_to_idx[b]]
    for a, b in adyacencia
    if a in node_to_idx and b in node_to_idx
]
edge_index = np.array(edge_index_list).T  # (2, n_conexiones)

print(f"edge_index shape: {edge_index.shape}")

edge_index shape: (2, 6864)


### 2.2. Cantidad de snapshots

Como construir un snapshot por cada hora del histórico cuesta mucho computacionalmente, lo vamos a muestrear. Vamos a usar 15.000 horas para train y 3.000 para test, usando el mismo corte temporal que usamos en LightGBM (últimos 6 meses como test).

In [9]:
con = duckdb.connect()
fecha_corte = "2025-10-30 01:00:00"

In [10]:
todas_horas_train = con.sql(f"""
    SELECT DISTINCT time_hour
    FROM '{data_clean}/dataset_model.parquet'
    WHERE time_hour < '{fecha_corte}'
""").df()["time_hour"].tolist()

todas_horas_test = con.sql(f"""
    SELECT DISTINCT time_hour
    FROM '{data_clean}/dataset_model.parquet'
    WHERE time_hour >= '{fecha_corte}'
""").df()["time_hour"].tolist()

print(f"Horas totales disponibles - train: {len(todas_horas_train)}, test: {len(todas_horas_test)}")

random.seed(42)
train_snapshots = 15000
test_snapshots = 3000

horas_train = random.sample(todas_horas_train, min(train_snapshots, len(todas_horas_train)))
horas_test = random.sample(todas_horas_test, min(test_snapshots, len(todas_horas_test)))

print(f"Horas muestreadas - train: {len(horas_train)}, test: {len(horas_test)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Horas totales disponibles - train: 57732, test: 4359
Horas muestreadas - train: 15000, test: 3000


Tenemos 57.732 horas disponibles en train y 4.359 en test.

### 2.3. LLevar los datos a esas horas

Hacemos una única consulta grande, filtrando por lo segmentos que forman parte del grafo de adyacencia.

In [11]:
nodos_con_vecinos_int = [int(x) for x in nodos_con_vecinos]

horas_train_str = "', '".join([str(h) for h in horas_train])
horas_test_str = "', '".join([str(h) for h in horas_test])
segmentos_str = ", ".join([str(x) for x in nodos_con_vecinos_int])

datos_train = con.sql(f"""
    SELECT segment_id, time_hour, avg_speed, speed_lag1,
           hora, dia_semana, mes, es_finde, festivo, school_holiday,
           temperature_2m, rain, snowfall, relative_humidity_2m, wind_speed_10m,
           partido, concierto, obra_tipo, accidente_tipo
    FROM '{data_clean}/dataset_model.parquet'
    WHERE time_hour IN ('{horas_train_str}')
      AND segment_id IN ({segmentos_str})
""").df()

datos_test = con.sql(f"""
    SELECT segment_id, time_hour, avg_speed, speed_lag1,
           hora, dia_semana, mes, es_finde, festivo, school_holiday,
           temperature_2m, rain, snowfall, relative_humidity_2m, wind_speed_10m,
           partido, concierto, obra_tipo, accidente_tipo
    FROM '{data_clean}/dataset_model.parquet'
    WHERE time_hour IN ('{horas_test_str}')
      AND segment_id IN ({segmentos_str})
""").df()

print(f"Filas train: {datos_train.shape}")
print(f"Filas test: {datos_test.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Filas train: (13081020, 19)
Filas test: (2649075, 19)


Conseguimos 13.087.728 filas en train y 2.648.770 filas en test.

### 2.4. Imputación para huecos

Como no todos los segmentos tienen datos en todas las horas, vamos a tener que rellenar esos huecos. Para ello, utilizamos la velocidad de referencia por segmento, día de la semana y hora (mediana histórica), es decir, la misma baseline que usamos para validar las ventanas de los eventos.

In [12]:
baseline = con.sql(f"""
    SELECT
        segment_id,
        EXTRACT(DOW FROM time_hour) AS dow,
        EXTRACT(HOUR FROM time_hour) AS hora,
        MEDIAN(avg_speed) AS speed_baseline
    FROM '{data_clean}/dataset_final.parquet'
    GROUP BY segment_id, EXTRACT(DOW FROM time_hour), EXTRACT(HOUR FROM time_hour)
""").df()

baseline_dict = {
    (row["segment_id"], row["dow"], row["hora"]): row["speed_baseline"]
    for _, row in baseline.iterrows()
}
print(f"Entradas en baseline: {len(baseline_dict)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Entradas en baseline: 171904


Tenemos 171.904 entradas en baseline.

### 2.5. Construir snapshots

Para cada nodo (segmento), si tiene datos reales a esa hora usa sus valores reales, y si no, rellenamos con el baseline y el contexto general de esa hora (clima, calendario, etc.). Guardamos qué nodos tenían dato real para luego evaluar el modelo solo sobre los nodos con dato real y no sobre los rellenados artificialmente.

De momento simplificamos `obra_tipo`/`accidente_tipo` a un indicador binario (hay obra/accidente sí o no), en vez de mantener las categorías de severidad, para no complicar el manejo dentro de la matriz numérica de la GCN.

In [13]:
col_features = [
    "speed_lag1", "hora", "dia_semana", "mes", "es_finde", "festivo", "school_holiday",
    "temperature_2m", "rain", "snowfall", "relative_humidity_2m", "wind_speed_10m",
    "partido", "concierto"
]

# obra_tipo y accidente_tipo son categóricas de texto, las convertimos a 0/1 simplificado por ahora
def construir_snapshot(grupo_hora, node_to_idx, baseline_dict, n_nodos):
    X = np.zeros((n_nodos, len(col_features) + 2))  # +2 para obra_activa, accidente_activo
    y = np.zeros(n_nodos)
    mask_tiene_dato = np.zeros(n_nodos, dtype=bool)

    contexto_general = grupo_hora.iloc[0]
    dow = contexto_general["dia_semana"]
    hora_val = contexto_general["hora"]

    filas_por_segmento = grupo_hora.set_index("segment_id").to_dict("index")

    for seg_id, idx in node_to_idx.items():
        if seg_id in filas_por_segmento:
            fila = filas_por_segmento[seg_id]
            X[idx] = [
                fila["speed_lag1"] if pd.notna(fila["speed_lag1"]) else baseline_dict.get((seg_id, dow, hora_val), 25.0),
                fila["hora"], fila["dia_semana"], fila["mes"], fila["es_finde"],
                fila["festivo"], fila["school_holiday"], fila["temperature_2m"],
                fila["rain"], fila["snowfall"], fila["relative_humidity_2m"], fila["wind_speed_10m"],
                fila["partido"], fila["concierto"],
                0 if fila["obra_tipo"] == "ninguna" else 1,
                0 if fila["accidente_tipo"] == "ninguno" else 1,
            ]
            y[idx] = fila["avg_speed"]
            mask_tiene_dato[idx] = True
        else:
            # segmento sin dato esta hora: rellenamos con baseline y contexto general
            X[idx] = [
                baseline_dict.get((seg_id, dow, hora_val), 25.0),
                hora_val, dow, contexto_general["mes"], contexto_general["es_finde"],
                contexto_general["festivo"], contexto_general["school_holiday"],
                contexto_general["temperature_2m"], contexto_general["rain"], contexto_general["snowfall"],
                contexto_general["relative_humidity_2m"], contexto_general["wind_speed_10m"],
                0, 0, 0, 0
            ]
            y[idx] = baseline_dict.get((seg_id, dow, hora_val), 25.0)
            mask_tiene_dato[idx] = False

    return X, y, mask_tiene_dato

Vamos a lanzar la contrucción sobre las 18.000 horas.

In [14]:
def construir_todos_snapshots(datos, horas, node_to_idx, baseline_dict, n_nodos):
    X_total, y_total, mask_total = [], [], []
    # agrupamos por hora y convertimos a diccionario
    datos_agrupados = dict(tuple(datos.groupby("time_hour")))

    for hora_actual in horas:
        if hora_actual not in datos_agrupados:
            continue
        grupo = datos_agrupados[hora_actual]
        X, y, mask = construir_snapshot(grupo, node_to_idx, baseline_dict, n_nodos)
        X_total.append(X)
        y_total.append(y)
        mask_total.append(mask)

    return np.array(X_total), np.array(y_total), np.array(mask_total)

print("Construyendo snapshots de train...")
X_train_gcn, y_train_gcn, mask_train_gcn = construir_todos_snapshots(
    datos_train, horas_train, node_to_idx, baseline_dict, n_nodos
)
print(f"X_train shape: {X_train_gcn.shape}")

print("Construyendo snapshots de test...")
X_test_gcn, y_test_gcn, mask_test_gcn = construir_todos_snapshots(
    datos_test, horas_test, node_to_idx, baseline_dict, n_nodos
)
print(f"X_test shape: {X_test_gcn.shape}")

Construyendo snapshots de train...
X_train shape: (15000, 1043, 16)
Construyendo snapshots de test...
X_test shape: (3000, 1043, 16)


Conseguimos 15.000 snapshots de train y 3.000 de test, cada uno con 1.043 nodos y 16 características.

In [15]:
# guardamos
np.savez_compressed(
    f"{data_clean}/gcn_snapshots.npz",
    X_train=X_train_gcn, y_train=y_train_gcn, mask_train=mask_train_gcn,
    X_test=X_test_gcn, y_test=y_test_gcn, mask_test=mask_test_gcn,
    edge_index=edge_index
)

node_to_idx_serializable = {str(int(k)): int(v) for k, v in node_to_idx.items()}
with open(f"{data_clean}/node_to_idx.json", "w") as f:
    json.dump(node_to_idx_serializable, f)

## 3. Preparación para entrenamiento

In [16]:
# cargamos los datos necesarios

datos_cargados = np.load(f"{data_clean}/gcn_snapshots.npz")
X_train_gcn = datos_cargados["X_train"]
y_train_gcn = datos_cargados["y_train"]
mask_train_gcn = datos_cargados["mask_train"]
X_test_gcn = datos_cargados["X_test"]
y_test_gcn = datos_cargados["y_test"]
mask_test_gcn = datos_cargados["mask_test"]
edge_index = datos_cargados["edge_index"]

with open(f"{data_clean}/node_to_idx.json") as f:
    node_to_idx = {int(k): v for k, v in json.load(f).items()}

n_nodos = len(node_to_idx)
print(f"Nodos: {n_nodos}")

Nodos: 1043


### 3.1. Normalizar las características

Las redes neuronales entrenan mejor cuando todas las variables están en rangos parecidos. Por lo tanto calculamos la media y desviación típica de cada característica usando train y los usamos para normalizar ambos conjuntos.

In [17]:
X_train_flat = X_train_gcn.reshape(-1, X_train_gcn.shape[-1])
feature_mean = X_train_flat.mean(axis=0)
feature_std = X_train_flat.std(axis=0) + 1e-8  # evitamos división por 0

X_train_norm = (X_train_gcn - feature_mean) / feature_std
X_test_norm = (X_test_gcn - feature_mean) / feature_std

### 3.2. Dataset y DataLoader

Cada snapshot representa un grafo completo:
- `x`: las características de los 1.043 nodos
- `edge_index`: las conexiones (iguales para todos los snapshots porque la red de calles es la misma)
- `y`: la velocidad real a predecir

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando: {device}")

edge_index_tensor = torch.tensor(edge_index, dtype=torch.long).to(device)
edge_index_cpu = edge_index_tensor.cpu()

Usando: cuda


In [19]:
class TrafficGraphDataset(Dataset):
    def __init__(self, X_norm, y, edge_index_cpu):
        self.X_norm = X_norm
        self.y = y
        self.edge_index = edge_index_cpu

    def __len__(self):
        return len(self.X_norm)

    def __getitem__(self, idx):
        return Data(
            x=torch.tensor(self.X_norm[idx], dtype=torch.float),
            edge_index=self.edge_index,
            y=torch.tensor(self.y[idx], dtype=torch.float)
        )

dataset_train = TrafficGraphDataset(X_train_norm, y_train_gcn, edge_index_cpu)
dataset_test = TrafficGraphDataset(X_test_norm, y_test_gcn, edge_index_cpu)

BATCH_SIZE = 32
loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
loader_test = DataLoader(dataset_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"Grafos train: {len(dataset_train)}, batches: {len(loader_train)}")

Grafos train: 15000, batches: 469


15.000 grafos de train y 469 batches de entrenamiento (15.000/32).

## 4. Definir la red GCN

Creamos tres capas de convolución de grafo (`GCNConv`) de 64 dimensiones con activación ReLU, y una capa lineal de salida.

`GCNConv` es la capa que propaga información espacial. Para cada nodo, combina sus propias características con las de sus vecinos directos (según `edge_index`). Con 3 capas apiladas, un nodo recibe influencia no solo de sus vecinos directos sino también de los vecinos de sus vecinos, hasta 3 saltos de distancia. Así podemos capturar cómo se propaga la congestión a lo largo de varias calles conectadas.

In [20]:
class TrafficGCN(nn.Module):
    def __init__(self, n_features, hidden_dim=64):
        super().__init__()
        self.conv1 = GCNConv(n_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.salida = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        return self.salida(x).squeeze(-1)

modelo_gcn = TrafficGCN(n_features=X_train_norm.shape[-1]).to(device)
optimizer = torch.optim.Adam(modelo_gcn.parameters(), lr=0.001)
print(modelo_gcn)

TrafficGCN(
  (conv1): GCNConv(16, 64)
  (conv2): GCNConv(64, 64)
  (conv3): GCNConv(64, 64)
  (salida): Linear(in_features=64, out_features=1, bias=True)
)


## 5. Entrenamiento

Usamos `L1Loss` (MAE) como función de pérdida, igual que en LightGBM.

In [21]:
criterio = nn.L1Loss()

def entrenar_epoca(modelo, loader, optimizer, criterio, device):
    modelo.train()
    total_loss = 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        pred = modelo(batch.x, batch.edge_index)
        loss = criterio(pred, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

def evaluar(modelo, loader, criterio, device):
    modelo.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = modelo(batch.x, batch.edge_index)
            loss = criterio(pred, batch.y)
            total_loss += loss.item() * batch.num_graphs
    return total_loss / len(loader.dataset)

Entrenamos hasta 50 épocas, midiendo el error en train y test después de cada una. Si el test_loss no mejora durante 8 épocas seguidas, se para. Cada vez que mejora el resultado en test guardamos el modelo, así al final nos quedamos con la versión que mejor generaliza, no necesariamente la última (que podría haber empezado a sobreajustar).

In [22]:
n_epocas = 50
mejor_test_loss = float("inf")
paciencia = 8
contador_paciencia = 0

for epoca in range(n_epocas):
    train_loss = entrenar_epoca(modelo_gcn, loader_train, optimizer, criterio, device)
    test_loss = evaluar(modelo_gcn, loader_test, criterio, device)
    print(f"Época {epoca+1}/{n_epocas} - train MAE: {train_loss:.4f} - test MAE: {test_loss:.4f}")

    if test_loss < mejor_test_loss:
        mejor_test_loss = test_loss
        contador_paciencia = 0
        torch.save(modelo_gcn.state_dict(), f"{data_clean}/modelo_gcn_mejor.pt")
    else:
        contador_paciencia += 1
        if contador_paciencia >= paciencia:
            print(f"Early stopping en época {epoca+1}")
            break

Época 1/50 - train MAE: 5.8568 - test MAE: 3.3320
Época 2/50 - train MAE: 3.2501 - test MAE: 3.0817
Época 3/50 - train MAE: 3.1297 - test MAE: 2.9862
Época 4/50 - train MAE: 3.0465 - test MAE: 2.8980
Época 5/50 - train MAE: 2.9628 - test MAE: 2.8152
Época 6/50 - train MAE: 2.8932 - test MAE: 2.7802
Época 7/50 - train MAE: 2.8539 - test MAE: 2.7396
Época 8/50 - train MAE: 2.8309 - test MAE: 2.7067
Época 9/50 - train MAE: 2.8187 - test MAE: 2.6952
Época 10/50 - train MAE: 2.8097 - test MAE: 2.6832
Época 11/50 - train MAE: 2.8033 - test MAE: 2.6768
Época 12/50 - train MAE: 2.7965 - test MAE: 2.6765
Época 13/50 - train MAE: 2.7913 - test MAE: 2.6715
Época 14/50 - train MAE: 2.7858 - test MAE: 2.6609
Época 15/50 - train MAE: 2.7798 - test MAE: 2.6584
Época 16/50 - train MAE: 2.7715 - test MAE: 2.6415
Época 17/50 - train MAE: 2.7580 - test MAE: 2.6230
Época 18/50 - train MAE: 2.7439 - test MAE: 2.6093
Época 19/50 - train MAE: 2.7320 - test MAE: 2.5997
Época 20/50 - train MAE: 2.7258 - test M

El early stopping no ha llegado a activarse, es decir, el modelo seguía mejorando, aunque fuera poco a poco, al llegar a la época 50. Conseguimos un MAE de 2.54km/h en test y 2.67km/h en train, algo pero que en el modelo de LightGBM.

## 6. Evaluación frente a LightGBM

La comparación con LightGBM todavía no es del todo justa. En cada snapshot, casi la mitad de los nodos no tenían dato real esa hora asiq ue se rellenaron con el baseline, no con el dato real.

Cuando evaluamos el error, el MAE se está calculando sobre todos los nodos, incluidos los rellenados artificialmente. Eso añade ruido a la métrica que no tiene que ver con si el modelo predice bien o mal, sino con cuánto se parece a un baseline que nunca vio como dato real.

En cambio, el MAE de LightGBM se calculó solo sobre filas con dato real. Por lo tanto, para comparar de forma justa, hay que evaluar la GCN sobre los nodos que tenían dato real en cada snapshot, usando la máscara que guardamos.

In [23]:
def evaluar_solo_reales(modelo, loader, mask_array, device):
    modelo.eval()
    errores = []
    idx_global = 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            pred = modelo(batch.x, batch.edge_index).cpu().numpy()
            y_real = batch.y.cpu().numpy()

            n_grafos_batch = batch.num_graphs
            n_nodos_grafo = mask_array.shape[1]

            for g in range(n_grafos_batch):
                mask_g = mask_array[idx_global]
                pred_g = pred[g*n_nodos_grafo:(g+1)*n_nodos_grafo]
                y_g = y_real[g*n_nodos_grafo:(g+1)*n_nodos_grafo]
                errores.extend(np.abs(pred_g[mask_g] - y_g[mask_g]))
                idx_global += 1

    return np.mean(errores)

mae_test_real = evaluar_solo_reales(modelo_gcn, loader_test, mask_test_gcn, device)
print(f"MAE de la GCN, solo sobre nodos con dato real: {mae_test_real:.4f} km/h")

MAE de la GCN, solo sobre nodos con dato real: 2.5764 km/h


Por lo tanto, el MAE de la GCN sobre los nodos con dato real es de 2.58km/h.

Con esto ya tenemos la comparación completa y justa entre los tres modelos:

| Modelo | MAE (km/h) | Qué usa |
|---|---|---|
| LightGBM principal | 1.85 | `segment_id` + `speed_lag1` + contexto |
| GCN (dependencias espaciales) | 2.58 | vecinos en el grafo + `speed_lag1` + contexto, sin `segment_id` |
| LightGBM de respaldo | 3.55 | `highway_type` + contexto, sin `segment_id` ni `speed_lag1` |

La GCN queda entre los otros dos modelos.Tiene sentido ya que no conoce la identidad exacta de cada segmento, la única formade saber dónde está es por la estructura del grafo y su `speed_lag1`.

Aun así supera al modelo de respaldo, que tampoco tiene identidad pero tampoco estructura espacial. Esto nos demuestra que la información de los segmentos vecinos aporta valor predictivo real, aunque no se conozca la identidad exacta del segmento.

En conclusión, la GCN no supera al modelo con información específica del segmento, pero sí valida la hipótesis de que la estructura espacial de la red de calles contiene información útil por sí misma.

In [24]:
# guardamos modelo final

torch.save({
    "model_state": modelo_gcn.state_dict(),
    "feature_mean": feature_mean,
    "feature_std": feature_std,
    "node_to_idx": node_to_idx,
    "n_features": X_train_norm.shape[-1]
}, f"{data_clean}/modelo_gcn_final.pt")